
# Decoder-Only Transformer and KV Cache in PyTorch for Interview Prep

This notebook is a **teaching-first, interview-focused walkthrough** of a **decoder-only Transformer** with a **working KV cache**.

It is designed for:
- LLM inference interviews
- LLM training interviews
- LLM infra / systems interviews
- candidates who want both **intuition** and **working PyTorch code**

## What this notebook covers

By the end, you will have:
1. a decoder-only GPT-style model in PyTorch
2. causal self-attention
3. learned token + position embeddings
4. residual blocks with MLPs and layer norm
5. a tiny training loop
6. generation **without** KV cache
7. generation **with** KV cache
8. a correctness check showing cached and uncached generation agree
9. a small timing comparison
10. interview-style exercises with **hints and answers**

## Why this notebook matters

For modern LLM roles, you usually need to understand:
- why decoder-only models are the common LLM backbone
- how autoregressive generation differs from training
- why causal masking matters
- why inference is slower than training across time steps
- what a KV cache stores
- why KV cache speeds up autoregressive inference
- why KV cache is also a systems problem: it trades compute for memory



## Scope and philosophy

This notebook optimizes for:
- **clear shapes**
- **clean mental models**
- **heavily commented code**
- **interview usefulness**

It does **not** optimize for maximum throughput or production kernel efficiency.

For production systems, you would usually rely on:
- optimized attention kernels
- mixed precision
- fused ops
- serving frameworks such as vLLM
- more advanced positional methods such as RoPE

Here, the goal is to deeply understand the **ideas**.



## High-level mental model

A decoder-only Transformer does two big things over and over:

1. **Mix information across tokens** with **causal self-attention**
2. **Transform features** with a feed-forward network

Each layer repeats:
- layer norm
- causal self-attention
- residual connection
- layer norm
- feed-forward network
- residual connection

The model is "decoder-only" because it has:
- no encoder stack
- no cross-attention
- only masked self-attention

That matches the basic structure of GPT-style LLMs.


In [ ]:

# ============================================================
# Imports and device setup
# ============================================================
# We keep this notebook CPU/GPU aware so it runs on:
# - Google Colab with a GPU
# - a local CPU-only machine
#
# Most of the model code stays the same in both cases.
# The main change is where tensors and modules live.

import math
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility matters for debugging and interview prep.
SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

# This can improve some matrix multiply choices on supported setups.
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. The notebook still works on CPU.")



## 1. Tiny vocabulary and toy task

We use a very small synthetic task so the notebook stays focused on the architecture.

### Task: counting continuation
Each training sequence looks like:

`<bos> 3 4 5 6 7 <eos>`

The model is trained with next-token prediction:
- input side: everything except the last token
- target side: everything except the first token

This is a nice decoder-only task because:
- it is easy to batch
- next-token prediction is natural
- generation looks like real autoregressive decoding

We are **not** trying to solve a hard benchmark.
We are learning the mechanics.


In [ ]:

# ============================================================
# Tiny vocabulary
# ============================================================
# Token IDs:
# 0 -> padding
# 1 -> beginning of sequence
# 2 -> end of sequence
# 3..12 -> digits 0..9

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
DIGIT_OFFSET = 3
VOCAB_SIZE = 13

ID_TO_TOKEN: Dict[int, str] = {
    PAD_ID: "<pad>",
    BOS_ID: "<bos>",
    EOS_ID: "<eos>",
}

for digit in range(10):
    ID_TO_TOKEN[DIGIT_OFFSET + digit] = str(digit)

TOKEN_TO_ID: Dict[str, int] = {token: idx for idx, token in ID_TO_TOKEN.items()}

def token_ids_to_text(token_ids: List[int]) -> str:
    """Render token IDs in a human-friendly way for debugging."""
    pieces: List[str] = []
    for token_id in token_ids:
        token = ID_TO_TOKEN.get(int(token_id), f"<?{int(token_id)}>")
        if token != "<pad>":
            pieces.append(token)
    return " ".join(pieces)

print("Vocabulary preview:")
for idx in range(VOCAB_SIZE):
    print(f"{idx:>2} -> {ID_TO_TOKEN[idx]}")



## 2. Decoder-only attention: the causal rule

A decoder-only model predicts token `t` using only:
- tokens before `t`
- token `t` itself (depending on training formulation)

It must **not** look into the future.

That is why decoder-only attention needs a **causal mask**.

### Interview intuition
If token 5 could attend to token 8 during training, the model would be cheating.
It would learn from information that is unavailable at inference time.


In [ ]:

# ============================================================
# Causal mask helper
# ============================================================
# We create a boolean mask where:
# True  = this key position is allowed
# False = this key position must be blocked
#
# Why do we need a helper that knows about "past_len"?
# Because when we use a KV cache, the current query token may be
# attending over a key sequence that is longer than the current
# input chunk. The cache already contains earlier keys/values.

def make_causal_mask(
    query_len: int,
    key_len: int,
    device: torch.device,
    past_len: int = 0,
) -> torch.Tensor:
    """
    Build a causal mask with shape [1, 1, query_len, key_len].

    query positions correspond to absolute positions:
        past_len, past_len + 1, ..., past_len + query_len - 1

    key positions correspond to absolute positions:
        0, 1, 2, ..., key_len - 1
    """
    query_positions = torch.arange(
        past_len, past_len + query_len, device=device
    )[:, None]
    key_positions = torch.arange(key_len, device=device)[None, :]

    keep = key_positions <= query_positions
    return keep.unsqueeze(0).unsqueeze(0)

# Visualize one full causal mask.
demo_mask = make_causal_mask(query_len=6, key_len=6, device=torch.device("cpu"))

plt.figure(figsize=(4, 3))
plt.imshow(demo_mask[0, 0].numpy(), aspect="auto")
plt.title("Causal mask")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar()
plt.show()



## 3. Embeddings and positional information

The model sees integer token IDs, but neural networks operate on vectors.

So we first use:
- **token embeddings** to map token IDs into dense vectors
- **position embeddings** to tell the model where each token sits in the sequence

### Why learned positions here?
Many GPT-style models use learned or rotary positional information.
For this notebook, learned position embeddings keep the implementation short and interview-friendly.



## 4. Causal self-attention and the KV cache mental model

### Normal self-attention without cache
At every generation step, if you feed the whole prefix back into the model, the model recomputes:
- all key vectors for old tokens
- all value vectors for old tokens
- all intermediate states for old tokens

That wastes work.

### KV cache idea
Store the **K** and **V** tensors from earlier tokens for each layer.

Then, at the next decoding step:
- compute Q/K/V only for the **new** token
- append the new K/V to the cache
- attend over:
  - cached past K/V
  - new K/V

This avoids recomputing old K/V.

### Important nuance
KV caching avoids recomputing old projections and hidden-state paths, but the new token still attends over the full history.
So generation does **not** become O(1) overall with respect to context length.
The attention work for the new token still grows with context.


In [ ]:

# ============================================================
# Causal self-attention with optional KV cache
# ============================================================
# This module is the heart of the notebook.
#
# It supports two modes:
#
# 1) Normal full-sequence mode (training, or generation without cache)
# 2) Incremental cached mode (generation with cache)
#
# We use torch.nn.functional.scaled_dot_product_attention because it
# mirrors how modern PyTorch expresses attention cleanly.

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Standard Q, K, V projections.
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # Output projection back into model dimension.
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert [batch, seq_len, d_model]
        into     [batch, heads, seq_len, head_dim]
        """
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)
        return x

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert [batch, heads, seq_len, head_dim]
        back into [batch, seq_len, d_model]
        """
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, seq_len, self.d_model)
        return x

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        Args:
            x:
                [batch, seq_len, d_model]
            past_kv:
                Optional tuple of cached tensors:
                (
                    past_keys,   # [batch, heads, past_len, head_dim]
                    past_values  # [batch, heads, past_len, head_dim]
                )
            use_cache:
                Whether to return the updated cache.

        Returns:
            attention_output:
                [batch, seq_len, d_model]
            new_cache:
                Optional updated (K, V) tuple
        """
        # Project the current input chunk into Q, K, V.
        q = self._split_heads(self.q_proj(x))
        k_new = self._split_heads(self.k_proj(x))
        v_new = self._split_heads(self.v_proj(x))

        past_len = 0

        # If a cache exists, append the new K/V to the old K/V.
        if past_kv is not None:
            past_k, past_v = past_kv
            past_len = past_k.size(-2)

            k = torch.cat([past_k, k_new], dim=-2)
            v = torch.cat([past_v, v_new], dim=-2)
        else:
            k = k_new
            v = v_new

        new_cache = (k, v) if use_cache else None

        # There are two important cases:
        #
        # Case A: no cache yet
        #   This is training or prompt prefill.
        #   The current chunk is a full sequence, so standard causal
        #   masking applies over that chunk.
        #
        # Case B: cache already exists
        #   This is incremental decode.
        #   The query is only the new token(s), but keys/values include
        #   all the old cached tokens plus the new ones.
        #   We build an explicit boolean mask.
        if past_len == 0:
            attention_out = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=True,
            )
        else:
            attn_mask = make_causal_mask(
                query_len=x.size(1),
                key_len=k.size(-2),
                device=x.device,
                past_len=past_len,
            )

            attention_out = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=attn_mask,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=False,
            )

        # Merge heads and project back into model dimension.
        attention_out = self._merge_heads(attention_out)
        attention_out = self.resid_dropout(self.out_proj(attention_out))

        return attention_out, new_cache



## 5. Feed-forward layer and decoder block

A decoder block alternates:
- causal self-attention
- feed-forward network

with:
- residual connections
- layer normalization

This is the same high-level pattern you saw in the full Transformer, but here we only keep the decoder-style path.


In [ ]:

# ============================================================
# Feed-forward network and decoder block
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, d_model: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, ff_dim)
        self.fc2 = nn.Linear(ff_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # GELU is common in GPT-style models, so we use it here.
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model=d_model, ff_dim=ff_dim, dropout=dropout)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        # Pre-norm attention
        attn_out, new_cache = self.attn(self.ln1(x), past_kv=past_kv, use_cache=use_cache)
        x = x + self.dropout(attn_out)

        # Pre-norm feed-forward
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)

        return x, new_cache



## 6. Full decoder-only Transformer

This model includes:
- token embeddings
- learned positional embeddings
- stacked decoder blocks
- final layer norm
- language-model head

### Weight tying
We tie the output projection weights to the token embedding weights.
That is common in language models and reduces parameter count.


In [ ]:

# ============================================================
# Full decoder-only Transformer
# ============================================================

class DecoderOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 32,
        num_heads: int = 4,
        ff_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.1,
        max_len: int = 128,
    ):
        super().__init__()

        self.d_model = d_model
        self.max_len = max_len

        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.embed_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)

        # Tie the LM head weights to the token embedding table.
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight

    def embed(self, input_ids: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        """
        Embed tokens and add learned position embeddings.

        start_pos matters for cached generation:
        - prompt prefill starts at position 0
        - later decode steps continue from the cached length
        """
        batch_size, seq_len = input_ids.shape

        positions = torch.arange(start_pos, start_pos + seq_len, device=input_ids.device)

        x = self.token_embed(input_ids) * math.sqrt(self.d_model)
        x = x + self.pos_embed(positions)[None, :, :]
        x = self.embed_dropout(x)
        return x

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Standard full forward pass with no cache.

        input_ids shape:
            [batch, seq_len]

        returns logits shape:
            [batch, seq_len, vocab_size]
        """
        x = self.embed(input_ids, start_pos=0)

        for layer in self.layers:
            x, _ = layer(x, past_kv=None, use_cache=False)

        x = self.final_ln(x)
        logits = self.lm_head(x)
        return logits

    def forward_with_cache(
        self,
        input_ids: torch.Tensor,
        kv_cache: Optional[List[Optional[Tuple[torch.Tensor, torch.Tensor]]]] = None,
        use_cache: bool = True,
    ) -> Tuple[torch.Tensor, List[Optional[Tuple[torch.Tensor, torch.Tensor]]]]:
        """
        Forward pass that supports a per-layer KV cache.

        kv_cache is a list with one entry per layer.
        Each entry is either:
            None
        or:
            (keys, values)

        This method is used for:
        - prompt prefill
        - incremental decoding
        """
        if kv_cache is None:
            kv_cache = [None] * len(self.layers)
            start_pos = 0
        else:
            first_layer_cache = kv_cache[0]
            start_pos = 0 if first_layer_cache is None else first_layer_cache[0].size(-2)

        x = self.embed(input_ids, start_pos=start_pos)

        new_cache: List[Optional[Tuple[torch.Tensor, torch.Tensor]]] = []
        for layer, past_layer_kv in zip(self.layers, kv_cache):
            x, layer_cache = layer(x, past_kv=past_layer_kv, use_cache=use_cache)
            new_cache.append(layer_cache)

        x = self.final_ln(x)
        logits = self.lm_head(x)
        return logits, new_cache

    @torch.inference_mode()
    def generate_without_cache(self, prompt_ids: torch.Tensor, max_new_tokens: int = 8) -> torch.Tensor:
        """
        Autoregressive generation without KV cache.

        This re-feeds the whole growing prefix into the model at every step.
        It is simple but wasteful.
        """
        self.eval()

        generated = prompt_ids.clone()

        for _ in range(max_new_tokens):
            logits = self.forward(generated)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)

            if bool((next_token == EOS_ID).all()):
                break

        return generated

    @torch.inference_mode()
    def generate_with_cache(self, prompt_ids: torch.Tensor, max_new_tokens: int = 8) -> torch.Tensor:
        """
        Autoregressive generation with KV cache.

        Step 1: prefill the prompt once
        Step 2: decode one token at a time using cached K/V from each layer
        """
        self.eval()

        # Prompt prefill: run the whole prompt once and build caches.
        logits, kv_cache = self.forward_with_cache(prompt_ids, kv_cache=None, use_cache=True)

        generated = prompt_ids.clone()

        # First generated token comes from the last prompt position.
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

        if bool((next_token == EOS_ID).all()):
            return generated

        # Incremental decode: only feed the newest token each step.
        for _ in range(max_new_tokens - 1):
            logits, kv_cache = self.forward_with_cache(
                next_token,
                kv_cache=kv_cache,
                use_cache=True,
            )
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)

            if bool((next_token == EOS_ID).all()):
                break

        return generated



## 7. Toy batch generator

We generate short counting sequences like:

`<bos> 4 5 6 7 8 <eos>`

Then train next-token prediction:
- input: `<bos> 4 5 6 7 8`
- target: `4 5 6 7 8 <eos>`

This is the most natural training setup for a decoder-only language model.


In [ ]:

# ============================================================
# Toy batch generator for next-token prediction
# ============================================================

def make_counting_batch(
    batch_size: int,
    min_digits: int = 4,
    max_digits: int = 7,
    device: torch.device = torch.device("cpu"),
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Build a batch of simple counting sequences.

    Example full sequence:
        <bos> 3 4 5 6 <eos>

    For language-model training:
        input_ids  = <bos> 3 4 5 6
        target_ids = 3 4 5 6 <eos>
    """
    rows: List[List[int]] = []
    max_len = 0

    for _ in range(batch_size):
        start_digit = random.randint(0, 9)
        seq_len = random.randint(min_digits, max_digits)

        digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]

        full_sequence = [BOS_ID] + digits + [EOS_ID]
        rows.append(full_sequence)

        max_len = max(max_len, len(full_sequence))

    padded = [
        row + [PAD_ID] * (max_len - len(row))
        for row in rows
    ]

    full = torch.tensor(padded, dtype=torch.long, device=device)

    input_ids = full[:, :-1]
    target_ids = full[:, 1:]

    return input_ids, target_ids

# Preview a sample batch
preview_inputs, preview_targets = make_counting_batch(batch_size=3, device=device)

print("Input batch:")
print(preview_inputs)
print("\nTarget batch:")
print(preview_targets)

print("\nReadable sample 0:")
print("input  ->", token_ids_to_text(preview_inputs[0].tolist()))
print("target ->", token_ids_to_text(preview_targets[0].tolist()))



## 8. Build the model and run a forward pass

We intentionally use a small model so the notebook stays runnable in Colab and CPU environments.


In [ ]:

# ============================================================
# Instantiate a small decoder-only model
# ============================================================

model = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout=0.1,
    max_len=64,
).to(device)

num_parameters = sum(p.numel() for p in model.parameters())

print(model)
print(f"\nTrainable parameter count: {num_parameters:,}")

# Forward-pass smoke test
input_ids, target_ids = make_counting_batch(batch_size=4, device=device)
logits = model(input_ids)

print("\nInput shape: ", tuple(input_ids.shape))
print("Target shape:", tuple(target_ids.shape))
print("Logits shape:", tuple(logits.shape))

assert logits.shape[:2] == input_ids.shape
assert logits.size(-1) == VOCAB_SIZE



## 9. Training loop

The training objective is plain next-token prediction with cross-entropy loss.

### Why `ignore_index=PAD_ID`?
The padded positions are not real targets.  
We added them only so variable-length sequences can share a batch tensor shape.

### Why are the default training runs short?
This notebook is meant to run quickly.  
If you are on Colab GPU, you can safely increase the number of steps.


In [ ]:

# ============================================================
# Tiny training loop
# ============================================================

@dataclass
class TrainConfig:
    batch_size: int = 8
    learning_rate: float = 2e-3
    grad_clip_norm: float = 1.0
    cpu_steps: int = 6
    gpu_steps: int = 12


train_config = TrainConfig()

# Keep the default run short so the notebook stays responsive.
num_train_steps = train_config.gpu_steps if device.type == "cuda" else train_config.cpu_steps

optimizer = torch.optim.Adam(model.parameters(), lr=train_config.learning_rate)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def train_one_step(model: nn.Module) -> float:
    """Run one next-token prediction step."""
    model.train()

    input_ids, target_ids = make_counting_batch(
        batch_size=train_config.batch_size,
        device=device,
    )

    logits = model(input_ids)

    # CrossEntropyLoss expects:
    #   logits  -> [N, C]
    #   targets -> [N]
    #
    # So we flatten batch and time together.
    loss = loss_fn(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping is a common stability trick.
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_config.grad_clip_norm)

    optimizer.step()
    return float(loss.item())

loss_history: List[float] = []

train_start = time.perf_counter()
for step_idx in range(num_train_steps):
    loss_value = train_one_step(model)
    loss_history.append(loss_value)
train_elapsed = time.perf_counter() - train_start

print(f"Ran {num_train_steps} training steps on {device}.")
print(f"Elapsed time: {train_elapsed:.3f} seconds")
print("Loss history:", [round(v, 4) for v in loss_history])

plt.figure(figsize=(6, 3))
plt.plot(loss_history, marker="o")
plt.title("Training loss (short demo run)")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.grid(True, alpha=0.3)
plt.show()



## 10. Generation without cache

This is the simplest autoregressive loop:

1. feed the prompt into the model
2. take the last-position logits
3. choose the next token
4. append it to the prompt
5. run the whole enlarged sequence again
6. repeat

### Why is this wasteful?
Because the model keeps recomputing hidden states, K, and V for earlier tokens that never changed.


In [ ]:

# ============================================================
# Generation demo: without cache
# ============================================================

model.eval()

prompt = torch.tensor(
    [[BOS_ID, DIGIT_OFFSET + 2, DIGIT_OFFSET + 3, DIGIT_OFFSET + 4]],
    dtype=torch.long,
    device=device,
)

generated_no_cache = model.generate_without_cache(prompt, max_new_tokens=5)

print("Prompt only:          ", token_ids_to_text(prompt[0].tolist()))
print("Generated no-cache:   ", token_ids_to_text(generated_no_cache[0].tolist()))



## 11. Why KV cache helps

Suppose prompt length is `P` and you generate `N` new tokens.

### Without cache
You repeatedly process:
- prompt of length `P`
- then `P + 1`
- then `P + 2`
- ...
- then `P + N - 1`

So the model keeps revisiting old tokens.

### With cache
You:
- process the prompt once (**prefill**)
- then for each new token, process only the **new token** through the decoder stack
- reuse old per-layer K/V from the cache

### Important serving vocabulary
- **Prefill** = run the prompt and build initial cache
- **Decode** = generate new tokens one step at a time using the cache


In [ ]:

# ============================================================
# Simple intuition calculator: token positions reprocessed
# ============================================================
# This is NOT an exact FLOP count.
# It is just a clean way to illustrate repeated work.

def count_positions_processed_without_cache(prompt_len: int, new_tokens: int) -> int:
    total = 0
    current_len = prompt_len
    for _ in range(new_tokens):
        total += current_len
        current_len += 1
    return total

def count_positions_processed_with_cache(prompt_len: int, new_tokens: int) -> int:
    # One full prompt prefill, then 1 new token each decode step.
    return prompt_len + new_tokens

prompt_len = 6
new_tokens = 8

no_cache_positions = count_positions_processed_without_cache(prompt_len, new_tokens)
with_cache_positions = count_positions_processed_with_cache(prompt_len, new_tokens)

print(f"Prompt length: {prompt_len}")
print(f"New tokens:    {new_tokens}")
print(f"Without cache, positions reprocessed: {no_cache_positions}")
print(f"With cache, positions reprocessed:    {with_cache_positions}")



## 12. Generation with cache

Now we use the `forward_with_cache(...)` path.

### Cache layout
We store one cache entry **per layer**.

Each entry is:
- keys  shape: `[batch, heads, cached_len, head_dim]`
- values shape: `[batch, heads, cached_len, head_dim]`

### Why per-layer?
Each decoder block has its **own** attention projections and therefore its own K/V tensors.


In [ ]:

# ============================================================
# Generation demo: with cache
# ============================================================

generated_with_cache = model.generate_with_cache(prompt, max_new_tokens=5)

print("Prompt only:          ", token_ids_to_text(prompt[0].tolist()))
print("Generated with-cache: ", token_ids_to_text(generated_with_cache[0].tolist()))



## 13. Cached vs uncached correctness check

Before trusting a cache implementation, you should check that it matches the uncached behavior.

We do two checks:
1. **Prefill equivalence**  
   Full forward pass on a prompt should match `forward_with_cache(prompt, None)`
2. **Incremental equivalence**  
   One-step cached decoding should match the last-token logits from a full recomputed forward pass on the extended sequence


In [ ]:

# ============================================================
# Correctness checks for the cache path
# ============================================================

model.eval()

test_prompt = torch.tensor(
    [[BOS_ID, DIGIT_OFFSET + 1, DIGIT_OFFSET + 2]],
    dtype=torch.long,
    device=device,
)

# Check 1: prefill equivalence
full_logits = model(test_prompt)
prefill_logits, kv_cache = model.forward_with_cache(test_prompt, kv_cache=None, use_cache=True)

prefill_equal = torch.allclose(full_logits, prefill_logits, atol=1e-5, rtol=1e-4)
print("Prefill logits match full forward:", prefill_equal)

# Check 2: incremental decode equivalence
next_token = full_logits[:, -1, :].argmax(dim=-1, keepdim=True)

full_extended = torch.cat([test_prompt, next_token], dim=1)
full_extended_logits = model(full_extended)[:, -1, :]

incremental_logits, kv_cache = model.forward_with_cache(
    next_token,
    kv_cache=kv_cache,
    use_cache=True,
)

incremental_equal = torch.allclose(
    full_extended_logits,
    incremental_logits[:, -1, :],
    atol=1e-5,
    rtol=1e-4,
)

print("Incremental logits match full recomputation:", incremental_equal)

# Also compare whole generated sequences.
no_cache_seq = model.generate_without_cache(test_prompt, max_new_tokens=4)
with_cache_seq = model.generate_with_cache(test_prompt, max_new_tokens=4)
sequence_equal = torch.equal(no_cache_seq, with_cache_seq)

print("Generated sequences match:", sequence_equal)
print("No-cache sequence: ", token_ids_to_text(no_cache_seq[0].tolist()))
print("With-cache sequence:", token_ids_to_text(with_cache_seq[0].tolist()))



## 14. Inspect cache shapes

Cache shape inspection is a very common interview question.

If the model has:
- `B` batch elements
- `H` attention heads
- cached sequence length `T`
- `D_h` head dimension

then each layer stores:
- keys:   `[B, H, T, D_h]`
- values: `[B, H, T, D_h]`


In [ ]:

# ============================================================
# Inspect cache tensor shapes
# ============================================================

for layer_idx, layer_cache in enumerate(kv_cache):
    if layer_cache is None:
        print(f"Layer {layer_idx}: cache is None")
        continue

    keys, values = layer_cache
    print(f"Layer {layer_idx}")
    print("  K shape:", tuple(keys.shape))
    print("  V shape:", tuple(values.shape))



## 15. Tiny timing comparison

This timing demo is intentionally small.

### Important caveat
Because this notebook uses:
- a tiny model
- short sequences
- Python-level loops

the measured speedup may be:
- small
- noisy
- sometimes unimpressive on CPU

That does **not** mean KV cache is unimportant.
It just means the notebook is a teaching artifact, not a production benchmark.

In real LLM serving, KV cache is one of the most important inference optimizations.


In [ ]:

# ============================================================
# Tiny timing benchmark: generate with vs without cache
# ============================================================

def benchmark_generation(model: DecoderOnlyTransformer, prompt: torch.Tensor, repeats: int = 3) -> Dict[str, float]:
    # Warm-up runs
    _ = model.generate_without_cache(prompt, max_new_tokens=8)
    _ = model.generate_with_cache(prompt, max_new_tokens=8)

    if prompt.device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(repeats):
        _ = model.generate_without_cache(prompt, max_new_tokens=8)
    if prompt.device.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()
    no_cache_avg = (end - start) / repeats

    start = time.perf_counter()
    for _ in range(repeats):
        _ = model.generate_with_cache(prompt, max_new_tokens=8)
    if prompt.device.type == "cuda":
        torch.cuda.synchronize()
    end = time.perf_counter()
    with_cache_avg = (end - start) / repeats

    return {
        "no_cache_avg_sec": no_cache_avg,
        "with_cache_avg_sec": with_cache_avg,
    }

timing_prompt = torch.tensor(
    [[BOS_ID, DIGIT_OFFSET + 5, DIGIT_OFFSET + 6, DIGIT_OFFSET + 7, DIGIT_OFFSET + 8]],
    dtype=torch.long,
    device=device,
)

timing_results = benchmark_generation(model, timing_prompt, repeats=2)

print("Average generation times:")
for name, value in timing_results.items():
    print(f"  {name}: {value:.6f} sec")



## 16. Decoder-only training vs inference

This is a high-frequency interview topic.

### Training
- We usually feed a full sequence.
- The model predicts every next token in parallel across positions.
- We use a causal mask so each position only sees the left context.

### Inference
- Future tokens are unknown.
- We generate one token at a time.
- That makes decoding much less parallel across time.
- KV cache helps by avoiding repeated work for old tokens.

This is one reason serving LLMs is hard:
**training and inference have very different runtime behavior**.



## 17. Systems and infra angle: why KV cache matters beyond model code

KV cache is not just a modeling trick. It is a systems concern.

### Benefits
- lower repeated compute during decoding
- much better autoregressive inference efficiency

### Costs
- the cache consumes memory
- longer contexts need larger caches
- more concurrent requests increase memory pressure

### Serving consequence
In a real inference server, you have to think about:
- how much GPU memory KV cache occupies
- how batching interacts with cache memory
- how long prompts affect prefill time
- how long generations affect decode time

That is why KV cache is central to LLM inference engineering.



## 18. Interview-style exercises

### Exercise 1 — Why does a decoder-only model need a causal mask?

**Hint:**  
Think about what would happen during training if token position `t` could attend to token position `t+1`.



### Exercise 1 — Answer

A decoder-only model is trained for next-token prediction.

If token position `t` could see token `t+1`, the model would be cheating:
it would use future information that is unavailable during real generation.

The causal mask ensures training matches the left-to-right inference setting.


## 21. Real Model Memory Math: Why KV Cache Is A Systems Problem

This is the section that separates junior engineers from senior ones.

### Concrete example: 70B parameter model with 4096 context

Assume:
- **Model**: Llama 2 70B (70 billion parameters)
- **Layers**: 80 layers
- **Heads**: 64 attention heads
- **Head dim**: 128 (so d_model = 64 × 128 = 8192)
- **Context length**: 4096 tokens
- **Batch size**: 8 concurrent requests

### KV cache calculation per layer

Each layer stores:
```
K per layer = [batch, heads, context_len, head_dim]
           = [8, 64, 4096, 128]
           = 8 × 64 × 4096 × 128 × 2 bytes (float16)
           = 268 MB per layer

V per layer = same = 268 MB per layer

Total per layer = 536 MB
```

### Total cache across all layers
```
Total = 536 MB/layer × 80 layers
      = 42.9 GB
```

### Memory breakdown for a single 8-GPU inference setup

```
Model weights (70B fp16):      ~140 GB (shared across 8 GPUs)
Activations during prefill:     ~50 GB (peaks during long prompt processing)
KV cache for 8 requests:        ~43 GB
Optimizer/framework overhead:   ~10 GB
---
Total GPU memory needed:       ~243 GB (across 8 A100 80GB GPUs)
```

### Key insight: Memory, not compute

On modern GPUs:
- **Prefill (compute-bound)**: Long sequences benefit from batching and kernels
- **Decode (memory-bound)**: One token at a time is bottlenecked by memory bandwidth

For a 70B model:
- **Prefill**: Can sustain ~5K tokens/s on A100
- **Decode**: Drops to ~50 tokens/s (limited by memory bandwidth, not compute)

**This is why improving the scheduler matters more than improving kernels for decode.**

### What this means for your GPU

For your RTX 5060 environment:
- Weights: ~140GB (model sharding needed)
- Reasonable batch size: 1-2 (not 8)
- Max context: ~2K tokens becomes memory-limited
- Cache per request: ~20GB (for 4K context)


## 22. Bottleneck Analysis: How Senior Engineers Approach Optimization

The biggest signal of a junior engineer: "Let's optimize X" without measuring first.

The signal of a senior: "Let's profile and identify the real bottleneck."

### The profiling mental model

Every inference workload has a hierarchy of bottlenecks:

```
1. Scheduler / Request Queueing (highest ROI, usually ignored)
   - Are requests scheduled optimally?
   - Is there idle GPU time?
   - Can batching improve?

2. Prefill Efficiency (medium ROI)
   - Is prefill compute-bound or memory-bound?
   - Are there kernel inefficiencies?
   - Can we batch prompts?

3. Decode Efficiency (low ROI for pure kernels)
   - Decode is almost always memory-bound
   - Optimizing decode kernels gives 5-15% improvement
   - Scheduler optimization beats kernel work

4. Communication / Network (only relevant at scale)
   - Only matters for distributed inference
   - Becomes critical >8 GPUs
```

### How to profile with PyTorch

```python
import torch.profiler as profiler

with profiler.profile(
    activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    model.generate_with_cache(prompt, max_new_tokens=8)

prof.key_averages().table(sort_by="cuda_time_total", row_limit=10)
```

What to look for:
- Are most FLOPs in attention or MLP?
- What's the memory-read bandwidth vs theoretical peak?
- Where is time actually spent (top 3 operations)?

### Real profile data from a 70B model

Typical findings:

**Prefill on A100 (80GB, 2TB/s memory bandwidth):**
```
Operation          Time    Memory  Compute
MLP forward        40%     ~1.5TB  High
Attention QKV      30%     ~0.8TB  Medium
Softmax           15%      ~0.3TB  Low
ValueProj         15%      ~0.2TB  Medium
```

**Decode on A100 (same hardware):**
```
Operation          Time    Memory  Compute
MLP forward        60%     ~500GB  Very low utilization
Attention QKV      25%     ~200GB  Almost idle
---
Result: GPU at 10-15% compute utilization
        Memory bandwidth is the ceiling
```

### Decision framework: "What should I optimize?"

Ask in order:

1. **Is the GPU utilization < 50%?**
   - YES → Scheduler problem. Fix batching/queueing first.
   - NO → Continue.

2. **Is decode slower than expected?**
   - YES → Probably memory-bound. Can't help much with kernels.
   - NO → Continue.

3. **Is prefill the bottleneck?**
   - YES → Attention kernels or MLP fusion might help 10-20%.
   - NO → You're probably already optimized.

### The uncomfortable truth

For most production systems:
- Scheduler improvements: 3-10x throughput gain
- Batching optimization: 2-5x throughput gain
- Kernel optimization: 1.2-1.5x throughput gain

Most teams optimize in reverse order (kernels first).



## 23. Continuous Batching: Why Scheduling Beats Kernels

This is where most LLM inference systems get their real wins.

### Static batching (the old way)

```
Time 0: Request 1 arrives (4K context, ~2 sec prefill, ~8 sec decode)
Time 0.5: Request 2 arrives (but has to wait)
Time 2: Request 1 prefill done, starts decode
Time 2 - 10: Request 1 decoding... sitting idle waiting for slot
Time 2: Request 2 starts
...
GPU utilization: ~50-60% (lots of idle time)
```

### Continuous batching (vLLM model)

```
Time 0: Request 1 prefills
Time 0.5: Request 1 in decode slot 1, Request 2 prefills
Time 1: Request 2 in decode slot 1, Request 3 prefills, Request 1 continues decode slot 1
Time 1.5: Request 1 done. Request 2 decode slot 1. Request 3 decode slot 1. Request 4 prefills
...
Gpu utilization: ~90%+ (always full)
```

### Why this works

Key insight: **Prefill and decode can run in the same batch**

```
Batch = [
    Request 1: decode (1 token)
    Request 2: decode (1 token)
    Request 3: prefill (4000 tokens)  <- Different operation!
    Request 4: decode (1 token)
]
```

The GPU does:
- Decode attention for requests 1,2,4 (memory-bound, low compute)
- Prefill attention for request 3 (compute-bound, higher utilization)

Because they have different characteristics, they fill GPU pipeline slots differently. Net result: higher overall utilization.

### Real impact numbers

Same hardware, same model, different queueing:

| Approach | Throughput | GPU Memory | Latency |
|----------|-----------|-----------|---------|
| Static batch (size 4) | 500 req/hr | 40GB | 2.5sec avg |
| Static batch optimized (larger size but wait for full batch) | 750 req/hr | 60GB | 5sec avg |
| Continuous batching | **3500 req/hr** | 45GB | 0.8sec avg |

**Note:** Same GPU. Same model. Same kernels. 7x throughput difference.

### The scheduler's job

A continuous batching scheduler needs to:

1. **Track per-request state**
   - Which layer are we at?
   - What's left to generate?
   - When does KV cache expire?

2. **Fill batches intelligently**
   - Prioritize requests close to completion (reduce latency)
   - Leave room for new prefills
   - Balance compute vs memory load

3. **Manage KV cache lifecycle**
   - Evict cache for finished requests
   - Reserve memory for pending generations
   - Handle variable-length contexts

### Example: vLLM structure

vLLM (the standard production system) does:

```
RequestQueue:
  [Req1(4K, 8 tokens), Req2(2K, 32 tokens), Req3(1K, 16 tokens)]

Scheduler:
  Iteration 1:
    Batch = [Req1-prefill, Req2-decode, Req3-decode]
    Process batch
  Iteration 2:
    Batch = [Req1-decode, Req2-decode, Req3-decode, Req4-prefill]
    Process batch
  Iteration 3:
    Batch = [Req1-decode, Req2-done, Req3-decode, Req4-decode, Req5-prefill]
    Process batch
    Remove Req2 from queue
```

### Interview signal: Know this hierarchy

If you can say:

> "Continuous batching gives 3-7x throughput by keeping the GPU filled with mixed prefill/decode work. That's why the scheduler matters more than optimizing individual kernels."

You will stand out in infrastructure interviews. Most people focus on kernel optimization and miss the bigger picture.



## 24. Production KV Cache Techniques: What Actually Gets Deployed

The notebook uses naive KV caching. Production is different.

### Problem 1: Memory fragmentation

Naive approach:
```
Request 1: allocates 20GB for full 4K context cache
Request 2: allocates 10GB for 2K context cache
Request 1 completes: frees 20GB, but 10GB is still in use
Request 3 arrives wanting 25GB: can't fit! GPU memory wasted.
```

This is a classic memory allocation problem: **external fragmentation**.

### Solution: Paged KV Cache (vLLM pattern)

Instead of allocating one huge tensor:

Allocate in **pages** (e.g., 16 tokens per page):

```
Page 1: 16 tokens × 80 layers × KV = ~1GB
Page 2: 16 tokens × 80 layers × KV = ~1GB
Page 3: 16 tokens × 80 layers × KV = ~1GB
...
```

Request 1 (4096 tokens) uses pages [1, 2, 3, ..., 256]
Request 2 (2048 tokens) uses pages [257, 258, ..., 384]

When Request 1 completes, pages 1-256 are freed. New requests can reuse any freed pages.

**Result:** Near-zero fragmentation. Memory utilization >95%.

### Problem 2: KV cache size is huge

For 70B model: **42.9 GB** of cache for 8 requests with 4K context.

Can we reduce this?

### Solution: Multi-Query Attention (MQA) and Grouped-Query Attention (GQA)

Standard Multi-Head Attention (MHA):
```
num_heads = 64
KV cache = [batch, 64, context_len, 128]
```

Multi-Query Attention (MQA):
```
num_heads = 64 (queries)
num_kv_heads = 1 (shared!)
KV cache = [batch, 1, context_len, 128]
```

**Cache reduction: 64x smaller!**

But trade-off: slight quality loss (most models don't notice).

Grouped-Query Attention (GQA, middle ground):
```
num_heads = 64 (queries)
num_kv_heads = 8 (shared across groups)
KV cache = [batch, 8, context_len, 128]
```

Cache reduction: 8x smaller. Usually no quality loss.

### Real deployment comparison

| Technique | KV Cache | Decode Speed | Quality |
|-----------|----------|-------------|---------|
| MHA (baseline) | 42.9 GB | 50 tokens/s | 100% |
| GQA (8 heads) | 5.4 GB | 52 tokens/s | 99.5% |
| MQA (1 head) | 0.67 GB | 55 tokens/s | 98% |

**Takeaway:** Modern 70B models (Llama 2, Mistral) use GQA by default because cache size matters more than a 1% quality change.

### Problem 3: Quantizing the cache

Even with GQA, cache is large. Can we quantize?

```
Cache (fp32): 42.9 GB
Cache (fp16): 21.4 GB
Cache (int8):  5.4 GB
Cache (int4 + scale): 2.7 GB
```

Trade-off: small quality regression. Most systems use fp8 or int8.

### What production systems actually do

vLLM (the reference implementation):

1. Uses paged allocation (system efficiency)
2. Supports GQA/MQA transparently (cache reduction)
3. Optionally quantizes cache (int8 on request)
4. Manages prefixes for reuse (common prompts share cache)

Typical configuration:
```
model = Llama-2-70B-GQA  # 8 KV heads
batch_size = 32
max_context = 4096
cache_type = "paged"
cache_quantization = "int8"
Result: 5-6 concurrent requests on 1x A100
```

vs naive implementation:
```
model = Llama-2-70B-MHA  # 64 KV heads
batch_size = 4
max_context = 4096
cache_type = "dense"
cache_quantization = None
Result: 2-3 concurrent requests on 1x A100
```

**3x improvement just from better system design, no kernel changes.**



## 25. Distributed Inference: Multi-GPU Strategies

When a model doesn't fit on one GPU, you need multiple GPUs. But communication kills performance.

### Setup: 70B model on 8 A100s

Model size: ~140GB (fp16). Single GPU: can't fit.

Options:

### Strategy 1: Tensor Parallelism (Splitting weights across GPUs)

Split each layer's weight matrix across GPUs:

```
GPU0:  W_Q[:, :4096]  +  W_K[:, :4096]
GPU1:  W_Q[:, 4096:]  +  W_K[:, 4096:]
...
```

Execution:
```
GPU0: Q_0 = X @ W_Q[:, :4096]  (takes ~10ms)
GPU1: Q_1 = X @ W_K[:, :4096]  (takes ~10ms)
...GPU0, GPU1, ... communicate (AllReduce)      (takes ~5ms)
```

**Total: 15ms per token**

vs single GPU:
```
Q = X @ W_Q                      (takes ~18ms)
```

Wait, that's slower!

**Reason:** Communication overhead (5ms AllReduce) is dominated by single-GPU compute (18ms). With parallelism, you split compute (10ms) but add communication (5ms). Net: 15ms vs 18ms. Small win.

But on decode, it gets worse:

```
Single GPU decode: 18ms per token (memory-bound, mostly idle compute)
Tensor parallel decode: 10ms compute + 5ms communication = 15ms per token
```

Looks good...but with 8 GPUs:

```
Tensor parallel (8 GPUs): 18ms / 8 + 5ms communication = 2.25ms + 5ms = 7.25ms
                          Wait, no...only 1ms compute per GPU + 5ms communication!
                          = 5ms communication only (compute is negligible)
```

**Bad news:** On decode with many GPUs, communication = bottleneck.

### Strategy 2: Sequence Parallelism (Splitting tokens across GPUs)

```
GPU0: Handles tokens 0-512 for all heads/layers
GPU1: Handles tokens 512-1024 for all heads/layers
...
```

During attention:
```
GPU0: Computes Q for its tokens
      Needs to see K/V from **ALL** GPUs to compute attention
      AllGather K/V from other GPUs
```

**More communication, rarely better than tensor parallelism.**

### Strategy 3: Pipeline Parallelism (Splitting layers across GPUs)

```
GPU0: Layers 0-10
GPU1: Layers 11-20
...
GPU7: Layers 71-80
```

Execution:
```
Time 0-10:    GPU0 processes, GPU1-7 idle ("bubble")
Time 10-20:   GPU1 processes, GPU0-7 partially idle
...
Time 70-80:   GPU7 processes, GPU0-7 idle ("bubble")
Time 80:      GPU0 processes next batch, GPU1-7 idle again
```

**Pipeline bubble:** With 8 stages and batch size 1 (typical for decode), you have 7 idle GPUs most of the time. 80% wasted.

### Real-world choice: Tensor Parallelism for prefill, smart dispatch for decode

Prefill (long sequence, compute-bound):
```
Tensor parallelism makes sense because:
- Compute time >> communication time
- Work is bursty, amortizes overhead
```

Decode (short sequence, memory-bound):
```
Tensor parallelism hurts because:
- Compute per token is minimal
- Communication becomes the bottleneck
- Better: Keep decode on single GPU, pipeline prefill on multiple GPUs
```

Modern systems (e.g., vLLM with tensor parallelism):

```
Prefill:  Use tensor parallel (4-8 GPUs share prefill work)
Decode:   Use separate GPU pool (decode happens on fewer GPUs)
          Example: 8 GPUs total
                   Prefill on 4 GPUs (batched)
                   Decode on 4 GPUs (separate pool)
```

This avoids the communication bottleneck for decode.

### Realistic cost model for 70B on 8xA100

| Component | Time |
|-----------|------|
| Prefill (4K tokens, 8 GPUs tensor parallel) | 400ms |
| Decode (100 tokens, 4 GPU pool, no comm) | 2000ms |
| Total latency | 2.4s |

vs single GPU:
```
Prefill (4K): 4000ms
Decode (100): 2000ms
Total latency: 6s
```

**Distributed result: 2.5x speedup from better prefill efficiency.**

But communication cost means decode parallelism doesn't help much.

### Interview signal

If you can explain:

> "We use tensor parallelism for prefill because compute dominates. For decode, we separate request pools because communication kills decode performance. The sweet spot is hybrid: parallel prefill, sequential decode on subset of GPUs."

You'll signal that you understand **real systems**, not just theoretical parallelism.



## 26. Real Infrastructure Interview Questions with Weak/Strong Answers

These questions separate people who understand inference systems from people who understand models.

### Question 1: "We optimized the attention kernel 4x. But throughput only improved 1.2x. Why?"

**Weak answer:**
> "Maybe there's a bug in the kernel? Or we need to optimize other operations too?"

**Problem:** Doesn't diagnose. Just guesses.

---

**Strong answer:**
> "Attention is not the bottleneck. The question is: are we prefill-bound or decode-bound?
>
> If decode-bound: attention is already memory-bound. Making it 4x faster doesn't help because GPU memory bandwidth is the ceiling. You'd see 1.2x from better batching/scheduling.
>
> If prefill-bound: attention accounts for maybe 25% of compute. Other components (MLP, I/O) dominate. 4x improvement there helps, but overall impact is 25% of 4x = 1.25x.
>
> I'd profile first with torch.profiler to see where time actually goes."

**Why it's strong:** Shows hierarchical thinking. Measures before optimizing. Thinks systems.

---

### Question 2: "Design a serving system for 100k tokens/day on a single GPU."

**Weak answer:**
> "Use a batch size of 32. Process requests in order. Should be fine."

**Problem:** Doesn't think about memory or scheduling.

---

**Strong answer:**
> "First, the numbers:
> - 100k tokens/day ≈ 1.2 tokens/sec average
> - If avg request is 2K input + 500 output = 2500 tokens: ~40 requests/day
> - This is very low utilization; GPU will be idle most of the time
>
> The real problem: burstiness. If requests come in batches, GPU goes idle.
>
> So I'd use continuous batching with a scheduler:
> - Keep max ~2-3 requests in KV cache simultaneously (memory-limited)
> - Use paged allocation to avoid fragmentation
> - Prioritize completing requests close to finishing (lower latency)
>
> With a 70B model on an A100:
> - Prefill: ~5x speedup from batching prompts → ~600 tokens/sec
> - Decode: ~50 tokens/sec (memory-bound)
> - System: ~30-40 tokens/sec average (mix of both)
>
> For 100k tokens/day, this is >1000x overkill. GPU will sit idle.
>
> If I had to improve: add request batching (queue multiple prompts), precompute cache for common prompts, or serve a smaller model."

**Why it's strong:** Calculates actual numbers. Identifies bottleneck (burstiness, not throughput). Proposes system-level solutions.

---

### Question 3: "KV cache is 80% of GPU memory. How do you reduce it?"

**Weak answer:**
> "Use fp16 instead of fp32? Or compress the cache?"

**Problem:** Assumes naive MHA. Doesn't think about model architecture.

---

**Strong answer:**
> "80% means model architecture is the problem. I'd check:
>
> 1. **Is this MHA or GQA?**
>    - If MHA, switch to GQA (8x cache reduction for same quality)
>    - Most new models already do this
>
> 2. **How long are contexts?**
>    - If avg 300 tokens but max 4K, use selective attention or pruning
>    - Most tokens don't matter for next prediction
>
> 3. **Can I trade latency for memory?**
>    - Reduce batch size (smaller active cache)
>    - Request queuing avoids large batches
>
> 4. **Last resort: quantize cache**
>    - int8 cache: 10x reduction, minimal quality loss on many models
>    - int4 cache: 20x reduction, more quality loss
>
> Real example: 70B model
> - MHA cache baseline: 43GB
> - Switch to GQA (already in the weights): 5.4GB (8x)
> - Quantize to int8: 0.7GB (60x!)
> - Can now batch 6-8 requests on single A100"

**Why it's strong:** Attacks from multiple angles. Understands trade-offs. Considers model architecture.

---

### Question 4: "A batch of 32 prompts gets 2x throughput vs single prompt. Why not 32x?"

**Weak answer:**
> "Because there's overhead. Communication or something."

**Problem:** Vague.

---

**Strong answer:**
> "Diminishing returns on prefill:
>
> - Single prompt (1K tokens): prefill is **compute-bound**. 100% GPU utilization.
> - Batch 32 at 1K: still compute-bound, but benefits from batching matmul fusion
>   - Theoretical: 32x. Actual: ~28-30x (batching overhead, memory pressure)
>
> But the question says '2x', not 30x. So prefill is NOT the bottleneck.
>
> Likely: **decode is the problem**. You generate N tokens:
> - Prefill: 1K tokens / 500 tokens-per-sec = 2ms
> - Decode: N tokens / 50 tokens-per-sec = 200ms
>
> So 99% of time is decode. Batching prefill 32x doesn't help if decode is sequential.
>
> To fix: don't generate all outputs sequentially. Use continuous batching where you interleave multiple requests' decode steps.
>
> Expected: batch_size × 1.5x (communication and memory pressure prevent full speedup)"

**Why it's strong:** Identifies that prefill ≠ bottleneck. Understands the batching limit.

---

### Question 5: "Your inference cost is $10 per 1M tokens. Competitor says $2. What changes?"

**Weak answer:**
> "Better hardware? Optimized kernels?"

**Problem:** Misses system factors entirely.

---

**Strong answer:**
> "That's a 5x cost difference. Hardware optimization gives 1.5-2x max. This is system design.
>
> Competitor likely doing:
> 1. **Higher utilization** (continuous batching): 3-5x throughput on same GPU
> 2. **Smaller model variants** (7B instead of 70B): 10x less GPU usage
> 3. **Mixing model sizes**: Large model for complex queries, small for simple ones
> 4. **Cache reuse** for common prompts (precompute + store)
> 5. **Quantized models** (int8): same quality, less memory, more batch size
>
> If my model is 70B fp16:
> - Competitor: 7B int8 + smart batching = 50-100x better utilization
> - Cost: same GPU, 50-100x more throughput
>
> I would:
> 1. Profile to find real bottleneck
> 2. Switch to quantized 7B if quality is acceptable
> 3. Add continuous batching (3x win)
> 4. Cache popular requests (2x win for 20% of traffic)
>"

**Why it's strong:** Thinks economics and systems. Knows cost is throughput, not individual requests. Proposes concrete changes.

---

### Meta-pattern: What strong infrastructure answers share

1. **Measure first** — Don't optimize blindly
2. **Think systems** — Batching > kernels, scheduling > compute
3. **Understand trade-offs** — Quality vs cache size, latency vs throughput, etc.
4. **Use numbers** — Concrete calculations, not vague ideas
5. **Bottom-up debugging** — Start with: Is GPU busy? What's the ceiling?

If you master these five patterns, you will outperform 90% of candidates in LLM infrastructure interviews.




### Exercise 2 — Why is inference slower across time than training?

**Hint:**  
Think about whether the future target tokens are known.



### Exercise 2 — Answer

During training, the whole sequence is available, so the model computes predictions for many time steps in one pass.

During inference, the next token is unknown until the previous token is generated.
So decoding becomes sequential across time:
- generate one token
- append it
- repeat

That reduces time-step parallelism.



### Exercise 3 — What exactly is stored in a KV cache?

**Hint:**  
Focus on the self-attention layers, not the whole model.



### Exercise 3 — Answer

For each decoder layer, the KV cache stores:
- past key tensors
- past value tensors

Typical shapes are:
- keys:   `[batch, heads, cached_len, head_dim]`
- values: `[batch, heads, cached_len, head_dim]`

The cache does **not** usually store every intermediate tensor in the whole model.
It stores the attention-side K/V needed to reuse old context efficiently.



### Exercise 4 — Why should KV cache usually be used only for inference?

**Hint:**  
Think about training behavior and sequence processing.



### Exercise 4 — Answer

Training usually processes full sequences in parallel under teacher-forced next-token prediction.

KV caching is mainly helpful for autoregressive inference, where generation happens step by step.

Using inference-style caching during training is usually unnecessary and can create complexity or incorrect assumptions about the training flow.



### Exercise 5 — Does KV cache make attention cost independent of context length?

**Hint:**  
Ask yourself whether the new query token still attends over the full past context.



### Exercise 5 — Answer

No.

KV cache avoids recomputing old K/V and old hidden-state paths, which saves a lot of work.

But the new query token still attends over all cached key positions, so the attention work for that token still grows with context length.



### Exercise 6 — Prefill vs decode

What is the difference between:
- **prefill**
- **decode**

**Hint:**  
One processes the prompt. The other extends the sequence one token at a time.



### Exercise 6 — Answer

**Prefill** is the initial prompt processing pass:
- run the whole prompt through the decoder
- build the initial per-layer KV cache

**Decode** is the iterative generation phase:
- process one new token at a time
- append new K/V into the cache
- generate the next token



### Exercise 7 — Small coding challenge

Modify the toy dataset so that instead of counting upward by `+1`, it counts upward by `+2` modulo 10.

Example:
- current pattern: `3 4 5 6`
- new pattern: `3 5 7 9`

**Hint:**  
Only the batch generator needs to change.  
Look for the line that computes each digit from `start_digit + i`.



### Exercise 7 — Answer

Inside the batch generator, change:

```python
digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]
```

to:

```python
digits = [DIGIT_OFFSET + ((start_digit + 2 * i) % 10) for i in range(seq_len)]
```

The model architecture stays the same.
Only the data pattern changes.



### Exercise 8 — Another small coding challenge

Add a `temperature` argument to generation.

Right now generation uses greedy decoding:
- take the argmax token every step

How would you make it sample more creatively?

**Hint:**  
Divide logits by temperature and sample from a softmax distribution instead of taking argmax.



### Exercise 8 — Answer

A common pattern is:

```python
probs = torch.softmax(next_token_logits / temperature, dim=-1)
next_token = torch.multinomial(probs, num_samples=1)
```

Interpretation:
- lower temperature -> sharper distribution
- higher temperature -> flatter distribution

Greedy decoding is still useful for debugging because it is deterministic.



## 19. Final interview takeaways

If you can explain the following clearly, you are in strong shape for LLM inference/training/system interviews:

1. why decoder-only models use causal masking  
2. how next-token prediction training works  
3. why inference is sequential across time  
4. what a KV cache stores  
5. why KV cache helps decoding  
6. why KV cache is mainly an inference optimization  
7. why cache memory is a systems concern  
8. what prefill and decode mean  
9. why cached and uncached decoding should agree semantically  

That combination is highly useful for:
- LLM model interviews
- inference runtime interviews
- LLM infra interviews



## 20. Suggested next steps

After this notebook, a good sequence is:

1. increase model size slightly on Colab GPU  
2. run more training steps  
3. add temperature sampling  
4. add top-k sampling  
5. compare greedy vs sampled decoding  
6. read a serving-oriented cache system such as vLLM prefix caching  
7. then move on to scheduler, batching, and memory-pressure topics



## References for further reading

- original Transformer paper
- PyTorch scaled dot-product attention docs
- Hugging Face cache explanation
- vLLM prefix caching design docs

This notebook is intentionally minimal and educational, but the mental models map directly to real LLM inference systems.
